# Training StyleGAN2 on CelebA Dataset

Source:<P>

https://github.com/lucidrains/stylegan2-pytorch

Adapted:<P>

Antonio Esteves @ UMinho, Jan 2025<P>

---
TODO:

* Modify `'OUR_WANDB_PROJECT_ID'`
* Modify `'OUR_WANDB_ENTITY'`
* In `../config/stylegan2_celeba_128x128_06.yaml` file, modify the hyperparameters `experiment_name`, `data_path`, `dataset_name`, `base_dir`.
---

**Table of Contents**

- Import the necessary libraries
- Load the configuration file
- Initializations and create necessary folders
- Login into Weights & Bias
- Track metadata and hyperparameters with Weights & Bias
- Helper classes
- Helper functions
- Attention mechanism
- Losses
- Create a custom Dataset from the images in a folder
- Augmentations
- StyleGAN2 classes
	- EqualizedLinear: Learning-rate equalized linear layer
	- StyleMapping: Style mapping network
	- Conv2dModulated: Convolution with weights modulation and demodulation
	- GeneratorBlock: Generator block
    - Generator
    - DiscriminatorBlock: Discriminator block
    - StyleGAN2 model
- Trainer class
- Training loop functions
- Train the StyleGAN2 model
- Conclusion

This notebook is about StyleGAN2 from the paper [Analyzing and Improving the Image Quality of StyleGAN](https://arxiv.org/pdf/1912.04958.pdf?ref=blog.paperspace.com). We will make a clean, simple, and readable implementation of StyleGAN2 using PyTorch, and try to replicate the original paper as closely as possible.

The dataset that we will use in this notebook is a balanced version of the [CelebA dataset](https://www.kaggle.com/datasets/jessicali9530/celeba-dataset), containing 168868 human faces cropped to 128*128 pixels.

## Import the Necessary Libraries

Let us start by loading all necessary dependencies.

First, we import `torch` since we will use PyTorch, and from there we import `nn` to model the networks, and we also import `optim`, a package that implements various optimization algorithms, such as sgd and Adam. From `torchvision` we import `transforms` to prepare the data and apply some transformations.

We also import `torch.nn.functional` as `F`, `DataLoader` and `Dataset` from `torch.utils.data` to create minibatches of samples from the dataset, `save_image` and `make_grid` from `torchvision.utils` to save images and create a grid of images, `log2` and `sqrt` form `math`, `numpy` for linear algebra, `os` for interaction with the operating system, `tqdm` to show progress bars, and `matplotlib.pyplot` to plot some images.

We import `wandb` por monitoring training with Weights and Biases platform, `yaml` to parse the configuration file, `time` for measuring training times, `natsorted` from `natsort` to sort files in a folder, `Path` from `pathlib` to handle folder paths, and `Image` from `PIL.image` to display images.

In [ ]:
import os
import wandb
import yaml
import json
import time
import torch

from   torch                        import nn, optim, einsum
from   torchvision                  import transforms
import torch.nn.functional          as     F
from   torch.utils.data             import DataLoader, Dataset
from   torch.autograd               import grad                    as torch_grad
from   torch.utils.data.distributed import DistributedSampler
from   torch.nn.parallel            import DistributedDataParallel as DDP
from   torchvision.utils            import save_image, make_grid
from   torchinfo                    import summary
from   pytorch_fid                  import fid_score

from   math                         import log2, sqrt, floor, ceil
from   natsort                      import natsorted
import numpy                        as     np
from   tqdm                         import tqdm         
import matplotlib.pyplot            as     plt
from   pathlib                      import Path
import PIL.Image                    as     Image
import pandas                       as     pd

import random
from   retry.api                    import retry_call
from   datetime                     import datetime
import torch.multiprocessing        as     mp
import torch.distributed            as     dist

from   tqdm                         import tqdm
from   math                         import floor, log2
from   random                       import random
from   shutil                       import rmtree
from   functools                    import partial
import multiprocessing
from   contextlib                   import contextmanager, ExitStack
from   einops                       import rearrange, repeat
from   kornia.filters               import filter2d
from   vector_quantize_pytorch      import VectorQuantize

try:
    from apex import amp
    APEX_AVAILABLE = True
except:
    APEX_AVAILABLE = False
%matplotlib inline

## Load the configuration file

- `experiment_name:           'StyleGAN2_CelebA_128x128_06'`    -> Name of the experiment.
- `dataset_name:              'CelebA balanced'`                -> Dataset used for training the models.
- `data_path:                 '.../datasets/celeba/balanced'` -> Root directory for dataset.
- `base_dir:                  '.../code/GANs'` -> Path to where the results and models folders will be created.
- `results_dir:               'results'` -> Folder where to save the generated images.
- `models_dir:                'models'`  -> Folder where to save the models.
- `load_trained_model:         False`    -> Load trained model from file (True) or not (False).
- `load_model_from:            None`     -> Path to the model to be loaded if 'load_trained_model=True'. 
- `image_size:                 128`      -> Spatial size of training images.
- `crop_size:                 128`       -> Size used to crop original images of the dataset.
- `filters_scaling_factor:     16`       -> Scaling factor used to obtain the number of filters in the model layers.
- `featuremaps_max:            512`      -> Maximum value of number of feature maps (or filters) to be considered.
- `transparent:                False`    -> Train with images in RGBA format containing an alpha layer (True) or not (False).
- `batch_size:                 32`       -> Batch size during training.
- `gradient_accumulate_every:  1`        -> Number of gradient calculations per iteration (to tradeoff a small batch size).
- `epochs:                     100`      -> Number of training epochs.
- `lr:                         0.0002`   -> Learning rate applied in the generator optimization.
- `lr_map_network:             0.1`      -> Learning rate applied in the mapping network optimization.
- `lr_disc_mult:               1.5`      -> Scaling factor applied to the generator LR to obtain the discriminator LR.
- `relative_disc_losses:       False`    -> Use discriminator losses relative to mean value of the discriminator output (True) or absolute losses (False).
- `num_workers:                None`     -> Number of workers used in the dataset DataLoader.
- `checkp_interval:            5`        -> Interval between saving model checkpoints (in epochs).
- `log_interval:               100`      -> Interval between successive logs (in steps).
- `evaluate_every_steps:       1000`     -> Interval between evaluations (in steps). 
- `generate:                   False`    -> Generate images ONLY with a pretrained model (True) or train the model (False).
- `grid_size:                  8`        -> At every evaluation step, generate a grid of images with size: grid_size x grid_size. 
- `num_grids_generate:         1`        -> Number of grids of images to generate at every evaluation step.     
- `generate_interpolation:     False`    -> Generate an interpolation video between two random points in latent space (True) or not (False).
- `interpolation_steps:        100`      -> Number of frames to consider when generating an interpolation video between two points in latent space.
- `save_frames:                False`    -> Save each frame of the interpolation video to a separate file (True) or not(False).
- `truncation_psi:             0.75`     -> Psi value used in style sampling truncation (psi=1 <=> complete image diversity, psi=0 <=> no diversity).
- `mixed_prob:                 0.9`      -> Probability of retrieving the latent Z vectors from a list that mixes two noise lists. ????  
- `fp16:                       False`    -> Use mixed precision (FP16 e FP32) in calculations (True) or only FP32 (False).
- `path_length_regulation:     True`     -> Apply a path length regularization term to the generator loss (True) or not (False).
- `contrastive_regulation:     False`    -> Apply a contrastive regularization term to the discriminator loss (True) or not (False).
- `latent_dim:                 512`      -> Latent vectors dimension.
- `vq_disc_layers:             []`       -> List with IDs of the discriminator layers to apply vector quantization. Example: [1,2].
- `vq_codebook_size:           256`      -> Number of possible values that the vectorization/discretization can assume.
- `attn_layers:                []`       -> List of IDs of discriminator layers to which self-attention is added after its output. Self-attention is also added to the symmetric layers of the generator. Example: [1,2].
- `initial_constant:           True`     -> Instantiate the initial constant at the generator beginning (True) or not (False).
- `aug_prob:                   0.0`      -> Probability of applying the augmentation transformations.
- `aug_types:                  ['translation', 'cutout']` -> List with the types of augmentations to apply.
- `generator_top_k:            False`    -> Zero out the gradient contribution from a percentage of samples classified as fake by the discriminator (True) or not (False). The idea is to improve generator optimization.
- `generator_top_k_gamma:      0.99`     -> Gamma value that controls how top-k decreases from using a full batch of samples to using a fraction of samples. 
- `generator_top_k_frac:       0.5`      -> Target fraction of sample to zero out during the top-k technique.
- `dual_contrastive_loss:      False`    -> Use contrastive loss between real and fake logits (True) or not (False).
- `dataset_aug_prob:           0.0`      -> Dataset data augmentation probability.
- `multi_gpus:                 False`    -> Train the model with multiple GPUs (True) or single GPU (False).
- `calculate_fid_interval:     None`     -> Interval between calculations of the FID metric. If set to 'None' FID is not calculated. 
- `calculate_fid_num_images:   12800`    -> Number of images to use during FID calculation.
- `clear_fid_cache:            False`    -> Remove existing files used in FID calculation and recreate directories (True) or not (False).
- `see:                        42`       -> Seed value for reproducible results.


In [ ]:
CONFIG_FILE = '../config/stylegan2_celeba_128x128_06.yaml'

with open(CONFIG_FILE, 'r') as file:
    try:
        config = yaml.safe_load(file)
    except yaml.YAMLError as exc:
        print(exc)

if (config['load_model_from'] == 'None'):
    config['load_model_from'] = None

if (config['num_workers'] == 'None'):
    config['num_workers'] = None

if (config['calculate_fid_interval'] == 'None'):
    config['calculate_fid_interval'] = None

if (config['crop_size'] == 'None'):
    config['crop_size'] = None

if config['transparent'] == True:
    config['channels'] = 4
else:
    config['channels'] = 3

## Initializations and create the necessary folders

In [ ]:
device                  = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Using {device} for computing')

train_dir    = Path(config["data_path"])

# Location where we will save here the images generated during WGAN-GP training
RESULTS_PATH = f'results/{config["experiment_name"]}'
os.makedirs(RESULTS_PATH, exist_ok=True)

# Location where the trained models will be saved
MODELS_PATH  = os.path.join(os.getcwd(), 'models')
os.makedirs(MODELS_PATH, exist_ok=True)

NUM_CORES = multiprocessing.cpu_count()
EXTS      = ['jpg', 'jpeg', 'png']

__version__ = '1.0_based_on_lucidrains_1.9.0'

## Login into Weights & Bias

In [ ]:
wandb.login()

## Track metadata and hyperparameters with Weights & Bias

Define the experiment: the hyperparameters, the dataset and model name. This information will be stored in a `config` dictionary.

In [ ]:
config_wandb = config

wandb.init(
    project = 'OUR_WANDB_PROJECT_ID',
    entity  = 'OUR_WANDB_ENTITY', 
    config  = config_wandb
)

### Format a given time in seconds as DD HH mm ss

In [ ]:
def time_format(seconds: int) -> str:
    '''
    Converts a time in seconds to days:hours:minutes:seconds.
    '''
    if seconds is not None:
        seconds = int(seconds)
        d = seconds // (3600 * 24)
        h = seconds // 3600 % 24
        m = seconds % 3600 // 60
        s = seconds % 3600 % 60
        if d > 0:
            return '{:02d}D {:02d}H {:02d}m {:02d}s'.format(d, h, m, s)
        elif h > 0:
            return '{:02d}H {:02d}m {:02d}s'.format(h, m, s)
        elif m > 0:
            return '{:02d}m {:02d}s'.format(m, s)
        elif s > 0:
            return '{:02d}s'.format(s)
    return '-'

## Helper classes

In [ ]:
class NanException(Exception):
    pass

class EMA():
    def __init__(self, beta):
        super().__init__()
        self.beta = beta
    def update_average(self, old, new):
        if not exists(old):
            return new
        return old * self.beta + (1 - self.beta) * new

class Flatten(nn.Module):
    def forward(self, x):
        return x.reshape(x.shape[0], -1)

class RandomApply(nn.Module):
    def __init__(self, prob, fn, fn_else = lambda x: x):
        super().__init__()
        self.fn      = fn
        self.fn_else = fn_else
        self.prob    = prob
    def forward(self, x):
        fn = self.fn if random() < self.prob else self.fn_else
        return fn(x)

class Residual(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn
    def forward(self, x):
        return self.fn(x) + x

class ChanNorm(nn.Module):
    def __init__(self, dim, eps = 1e-5):
        super().__init__()
        self.eps = eps
        self.g   = nn.Parameter(torch.ones(1, dim, 1, 1))
        self.b   = nn.Parameter(torch.zeros(1, dim, 1, 1))

    def forward(self, x):
        var  = torch.var(x, dim = 1, unbiased = False, keepdim = True)
        mean = torch.mean(x, dim = 1, keepdim = True)
        return (x - mean) / (var + self.eps).sqrt() * self.g + self.b

class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.fn   = fn
        self.norm = ChanNorm(dim)

    def forward(self, x):
        return self.fn(self.norm(x))

class PermuteToFrom(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn
    def forward(self, x):
        x             = x.permute(0, 2, 3, 1)
        out, *_, loss = self.fn(x)
        out           = out.permute(0, 3, 1, 2)
        return out, loss

class Blur(nn.Module):
    def __init__(self):
        super().__init__()
        f = torch.Tensor([1, 2, 1])
        self.register_buffer('f', f)
    def forward(self, x):
        f = self.f
        f = f[None, None, :] * f [None, :, None]
        return filter2d(x, f, normalized=True)

## Helper functions

In [ ]:
def exists(val):
    return val is not None

@contextmanager
def null_context():
    yield

def combine_contexts(contexts):
    @contextmanager
    def multi_contexts():
        with ExitStack() as stack:
            yield [stack.enter_context(ctx()) for ctx in contexts]
    return multi_contexts

def default(value, d):
    return value if exists(value) else d

def cycle(iterable):
    while True:
        for i in iterable:
            yield i

def cast_list(el):
    return el if isinstance(el, list) else [el]

def is_empty(t):
    if isinstance(t, torch.Tensor):
        return t.nelement() == 0
    return not exists(t)

def raise_if_nan(t):
    if torch.isnan(t):
        raise NanException

def gradient_accumulate_contexts(gradient_accumulate_every, is_ddp, ddps):
    if is_ddp:
        num_no_syncs = gradient_accumulate_every - 1
        head = [combine_contexts(map(lambda ddp: ddp.no_sync, ddps))] * num_no_syncs
        tail = [null_context]
        contexts =  head + tail
    else:
        contexts = [null_context] * gradient_accumulate_every

    for context in contexts:
        with context():
            yield

def loss_backwards(fp16, loss, optimizer, loss_id, **kwargs):
    if fp16:
        with amp.scale_loss(loss, optimizer, loss_id) as scaled_loss:
            scaled_loss.backward(**kwargs)
    else:
        loss.backward(**kwargs)

def gradient_penalty(images, output, weight = 10, center = 0.):
    batch_size = images.shape[0]
    gradients = torch_grad(outputs=output, inputs=images,
                           grad_outputs=torch.ones(output.size(), device=images.device),
                           create_graph=True, retain_graph=True, only_inputs=True)[0]

    gradients = gradients.reshape(batch_size, -1)
    return weight * ((gradients.norm(2, dim=1) - center) ** 2).mean()

def calc_pl_lengths(styles, images):
    device = images.device
    num_pixels = images.shape[2] * images.shape[3]
    pl_noise = torch.randn(images.shape, device=device) / sqrt(num_pixels)
    outputs = (images * pl_noise).sum()

    pl_grads = torch_grad(outputs=outputs, inputs=styles,
                          grad_outputs=torch.ones(outputs.shape, device=device),
                          create_graph=True, retain_graph=True, only_inputs=True)[0]

    return (pl_grads ** 2).sum(dim=2).mean(dim=1).sqrt()

def noise(n, latent_dim, device):
    '''
    Returns a tensor with size [n x latent_dim] filled with random numbers from a normal distribution N(0,1).
    '''
    return torch.randn(n, latent_dim).to(device)

def noise_list(n, layers, latent_dim, device):
    '''
    Returns a list containing a tuple (noise,#layers), where "noise" is a tensor with
    size [n x latent_dim] filled with random numbers from a normal distribution N(0,1).
    '''
    return [(noise(n, latent_dim, device), layers)]

def mixed_list(n, layers, latent_dim, device):
    tt = int(torch.rand(()).numpy() * layers)
    return noise_list(n, tt, latent_dim, device) + noise_list(n, layers - tt, latent_dim, device)

def latent_to_w(style_vectorizer, latent_descr):
    '''
    Map a latent tensor z (included in 'latent_descr') to an intermediate latent tensor w,
    using the style mapping network 'style_vectorizer'.
    '''
    return [(style_vectorizer(z), num_layers) for z, num_layers in latent_descr]

def image_noise(n, img_size, device):
    '''
    Returns 'n' noise tensors uniformelly distributed, each with size 'img_size x img_size'.
    '''
    return torch.FloatTensor(n, img_size, img_size, 1).uniform_(0., 1.).cuda(device)

def leaky_relu(p=0.2):
    return nn.LeakyReLU(p, inplace=True)

def evaluate_in_chunks(max_batch_size, model, *args):
    '''
    Evaluates the model with the data divided in chuncks of size 'max_batch_size'.
    '''
    split_args = list(zip(*list(map(lambda x: x.split(max_batch_size, dim=0), args))))
    chunked_outputs = [model(*i) for i in split_args]
    if len(chunked_outputs) == 1:
        return chunked_outputs[0]
    return torch.cat(chunked_outputs, dim=0)

def styles_def_to_tensor(styles_def):
    return torch.cat([t[:, None, :].expand(-1, n, -1) for t, n in styles_def], dim=1)

def set_requires_grad(model, bool):
    for p in model.parameters():
        p.requires_grad = bool

def slerp(val, low, high):
    low_norm  = low / torch.norm(low, dim=1, keepdim=True)
    high_norm = high / torch.norm(high, dim=1, keepdim=True)
    omega     = torch.acos((low_norm * high_norm).sum(1))
    so        = torch.sin(omega)
    res       = (torch.sin(
        (1.0 - val) * omega) / so).unsqueeze(1) * low + (torch.sin(val * omega) / so
    ).unsqueeze(1) * high
    return res

def cast_list(el):
    '''
    Convertd 'el' to a List if it is not yet a list.
    '''
    return el if isinstance(el, list) else [el]

def timestamped_filename(prefix = 'generated-'):
    '''
    Obtains a string containing "generated-" followed by the current date and time.
    '''
    now       = datetime.now()
    timestamp = now.strftime("%m-%d-%Y_%H-%M-%S")
    return f'{prefix}{timestamp}'

def set_seed(seed):
    '''
    Set random seeds for reproducible results.
    '''
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    np.random.seed(seed)
    random.seed(seed)

## Attention mechanism

In [ ]:
class DepthWiseConv2d(nn.Module):

    def __init__(
            self,
            dim_in,
            dim_out,
            kernel_size,
            padding = 0,
            stride  = 1,
            bias    = True
        ):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(
                dim_in,
                dim_in,
                kernel_size = kernel_size,
                padding     = padding,
                groups      = dim_in,
                stride      = stride,
                bias        = bias,
            ),
            nn.Conv2d(
                dim_in,
                dim_out,
                kernel_size = 1,
                bias        = bias,
            )
        )
    def forward(self, x):
        return self.net(x)

class LinearAttention(nn.Module):

    def __init__(self, dim, dim_head = 64, heads = 8):
        super().__init__()
        self.scale  = dim_head ** -0.5
        self.heads  = heads
        inner_dim   = dim_head * heads

        self.nonlin = nn.GELU()
        self.to_q   = nn.Conv2d(dim, inner_dim, 1, bias = False)
        self.to_kv  = DepthWiseConv2d(dim, inner_dim * 2, 3, padding = 1, bias = False)
        self.to_out = nn.Conv2d(inner_dim, dim, 1)

    def forward(self, fmap):
        h, x, y = self.heads, *fmap.shape[-2:]
        q, k, v = (self.to_q(fmap), *self.to_kv(fmap).chunk(2, dim = 1))
        q, k, v = map(lambda t: rearrange(t, 'b (h c) x y -> (b h) (x y) c', h = h), (q, k, v))

        q       = q.softmax(dim = -1)
        k       = k.softmax(dim = -2)

        q       = q * self.scale

        context = einsum('b n d, b n e -> b d e', k, v)
        out     = einsum('b n d, b d e -> b n e', q, context)
        out     = rearrange(out, '(b h) (x y) d -> b (h d) x y', h = h, x = x, y = y)

        out     = self.nonlin(out)
        return self.to_out(out)

# A layer of self-attention and feedforward for images.

attn_and_ff = lambda chan: nn.Sequential(
    *[
        Residual(
            PreNorm(chan, LinearAttention(chan))
        ),
        Residual(
            PreNorm(
                chan,
                nn.Sequential(
                    nn.Conv2d(chan, chan * 2, 1),
                    leaky_relu(),
                    nn.Conv2d(chan * 2, chan, 1),
                )
            )
        )
    ]
)

## Losses

In [ ]:
def gen_hinge_loss(fake, real):
    return fake.mean()

def hinge_loss(fake, real):
    return (F.relu(1 + real) + F.relu(1 - fake)).mean()

def dual_contrastive_loss(fake_logits, real_logits):
    device = real_logits.device
    real_logits, fake_logits = map(
        lambda t: rearrange(t, '... -> (...)'),
        (real_logits,fake_logits),
    )

    def loss_half(t1, t2):
        t1 = rearrange(t1, 'i -> i 1')
        t2 = repeat(t2, 'j -> i j', i = t1.shape[0])
        t  = torch.cat((t1, t2), dim = -1)
        return F.cross_entropy(t, torch.zeros(t1.shape[0], device = device, dtype = torch.long))

    return loss_half(real_logits, fake_logits) + loss_half(-fake_logits, -real_logits)

## Create a custom Dataset from the images in a folder

In [ ]:
def convert_rgb_to_transparent(image):
    if image.mode != 'RGBA':
        return image.convert('RGBA')
    return image

def convert_transparent_to_rgb(image):
    if image.mode != 'RGB':
        return image.convert('RGB')
    return image

class expand_greyscale(object):
    def __init__(self, transparent):
        self.transparent = transparent

    def __call__(self, tensor):
        channels = tensor.shape[0]
        num_target_channels = 4 if self.transparent else 3

        if channels == num_target_channels:
            return tensor

        alpha = None
        if channels == 1:
            color = tensor.expand(3, -1, -1)
        elif channels == 2:
            color = tensor[:1].expand(3, -1, -1)
            alpha = tensor[1:]
        else:
            raise Exception(f'Image with invalid number of channels given {channels}')

        if not exists(alpha) and self.transparent:
            alpha = torch.ones(1, *tensor.shape[1:], device=tensor.device)

        return color if not self.transparent else torch.cat((color, alpha))

def resize_to_minimum_size(min_size, image):
    if max(*image.size) < min_size:
        return transforms.functional.resize(image, min_size)
    return image

In [ ]:
class CustomDataSet(Dataset):

    def __init__(self, root_dir, transform, transparent, aug_prob):
        self.root_dir     = root_dir
        self.transform    = transform
        self.transparent  = transparent
        self.aug_prob     = aug_prob
        self.all_images   = os.listdir(root_dir)
        self.total_images = natsorted(self.all_images)

    def __len__(self):
        return len(self.total_images)

    def __getitem__(self, idx):
        img_loc      = os.path.join(self.root_dir, self.total_images[idx])
        image        = Image.open(img_loc).convert("RGB")
        tensor_image = self.transform(image)
        return tensor_image

## Augmentations

Augmentation functions got images as `x`, where `x` is a tensor with these dimensions:
- 0 - count of images
- 1 - channels
- 2 - width of image
- 3 - height of image


In [ ]:
def rand_brightness(x, scale):
    x = x + (torch.rand(
        x.size(0),
        1,
        1,
        1,
        dtype  = x.dtype,
        device = x.device,
    ) - 0.5) * scale
    return x

def rand_saturation(x, scale):
    x_mean = x.mean(dim=1, keepdim=True)
    x = (x - x_mean) * (((torch.rand(
        x.size(0),
        1,
        1,
        1,
        dtype  = x.dtype,
        device = x.device,
    ) - 0.5) * 2.0 * scale) + 1.0) + x_mean
    return x

def rand_contrast(x, scale):
    x_mean = x.mean(dim=[1, 2, 3], keepdim=True)
    x = (x - x_mean) * (((torch.rand(
        x.size(0),
        1,
        1,
        1,
        dtype  = x.dtype,
        device = x.device,
    ) - 0.5) * 2.0 * scale) + 1.0) + x_mean
    return x

def rand_translation(x, ratio=0.125):
    shift_x, shift_y = int(x.size(2) * ratio + 0.5), int(x.size(3) * ratio + 0.5)
    translation_x = torch.randint(
        -shift_x,
        shift_x + 1,
        size=[x.size(0), 1, 1],
        device=x.device,
    )
    translation_y = torch.randint(
        -shift_y, shift_y + 1,
        size=[x.size(0), 1, 1],
        device=x.device,
    )
    grid_batch, grid_x, grid_y = torch.meshgrid(
        torch.arange(x.size(0), dtype=torch.long, device=x.device),
        torch.arange(x.size(2), dtype=torch.long, device=x.device),
        torch.arange(x.size(3), dtype=torch.long, device=x.device),
    )
    grid_x = torch.clamp(grid_x + translation_x + 1, 0, x.size(2) + 1)
    grid_y = torch.clamp(grid_y + translation_y + 1, 0, x.size(3) + 1)
    x_pad = F.pad(x, [1, 1, 1, 1, 0, 0, 0, 0])
    x = x_pad.permute(0, 2, 3, 1).contiguous()[grid_batch, grid_x, grid_y].permute(0, 3, 1, 2)
    return x

def rand_offset(x, ratio=1, ratio_h=1, ratio_v=1):
    w, h = x.size(2), x.size(3)

    imgs = []
    for img in x.unbind(dim = 0):
        max_h = int(w * ratio * ratio_h)
        max_v = int(h * ratio * ratio_v)

        value_h = random.randint(0, max_h) * 2 - max_h
        value_v = random.randint(0, max_v) * 2 - max_v

        if abs(value_h) > 0:
            img = torch.roll(img, value_h, 2)

        if abs(value_v) > 0:
            img = torch.roll(img, value_v, 1)

        imgs.append(img)

    return torch.stack(imgs)

def rand_offset_h(x, ratio=1):
    return rand_offset(x, ratio=1, ratio_h=ratio, ratio_v=0)

def rand_offset_v(x, ratio=1):
    return rand_offset(x, ratio=1, ratio_h=0, ratio_v=ratio)

def rand_cutout(x, ratio=0.5):
    cutout_size = int(x.size(2) * ratio + 0.5), int(x.size(3) * ratio + 0.5)
    offset_x    = torch.randint(0, x.size(2) + (1 - cutout_size[0] % 2), size=[x.size(0), 1, 1], device=x.device)
    offset_y    = torch.randint(0, x.size(3) + (1 - cutout_size[1] % 2), size=[x.size(0), 1, 1], device=x.device)
    grid_batch, grid_x, grid_y = torch.meshgrid(
        torch.arange(x.size(0), dtype=torch.long, device=x.device),
        torch.arange(cutout_size[0], dtype=torch.long, device=x.device),
        torch.arange(cutout_size[1], dtype=torch.long, device=x.device),
    )
    grid_x = torch.clamp(grid_x + offset_x - cutout_size[0] // 2, min=0, max=x.size(2) - 1)
    grid_y = torch.clamp(grid_y + offset_y - cutout_size[1] // 2, min=0, max=x.size(3) - 1)
    mask   = torch.ones(x.size(0), x.size(2), x.size(3), dtype=x.dtype, device=x.device)
    mask[grid_batch, grid_x, grid_y] = 0
    x      = x * mask.unsqueeze(1)
    return x
def random_hflip(tensor, prob):
    if prob < random():
        return tensor
    return torch.flip(tensor, dims=(3,))

def DiffAugment(x, types=[]):
    for p in types:
        for f in AUGMENT_FNS[p]:
            x = f(x)
    return x.contiguous()

class AugWrapper(nn.Module):
    def __init__(self, D, image_size):
        super().__init__()
        self.D = D

    def forward(
            self,
            images,
            prob                = 0.,
            types               = [],
            detach              = False,
            return_aug_images   = False,
            input_requires_grad = False,
        ):
        if random() < prob:
            images = random_hflip(images, prob=0.5)
            images = DiffAugment(images, types=types)

        if detach:
            images = images.detach()

        if input_requires_grad:
            images.requires_grad_()

        logits = self.D(images)

        if not return_aug_images:
            return logits

        return images, logits

AUGMENT_FNS = {
    'brightness':      [partial(rand_brightness, scale=1.)],
    'lightbrightness': [partial(rand_brightness, scale=.65)],
    'contrast':        [partial(rand_contrast, scale=.5)],
    'lightcontrast':   [partial(rand_contrast, scale=.25)],
    'saturation':      [partial(rand_saturation, scale=1.)],
    'lightsaturation': [partial(rand_saturation, scale=.5)],
    'color':           [partial(rand_brightness, scale=1.), partial(rand_saturation, scale=1.), partial(rand_contrast, scale=0.5)],
    'lightcolor':      [partial(rand_brightness, scale=0.65), partial(rand_saturation, scale=.5), partial(rand_contrast, scale=0.5)],
    'offset':          [rand_offset],
    'offset_h':        [rand_offset_h],
    'offset_v':        [rand_offset_v],
    'translation':     [rand_translation],
    'cutout':          [rand_cutout],
}

## StyleGAN2 classes

In [ ]:
class EqualizedLinear(nn.Module):
    '''
    Learning-rate equalized linear layer.
    '''
    def __init__(self, in_dim, out_dim, lr_mul = 1, bias = True):
        super().__init__()
        self.weight   = nn.Parameter(torch.randn(out_dim, in_dim))
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_dim))

        self.lr_mul = lr_mul

    def forward(self, input):
        return F.linear(
            input,
            weight = self.weight * self.lr_mul,
            bias   = self.bias * self.lr_mul,
        )

class StyleMapping(nn.Module):
    '''
    Style mapping network.
    '''
    def __init__(self, emb, depth, lr_mul = 0.1):
        super().__init__()

        layers = []
        for i in range(depth):
            layers.extend([EqualizedLinear(emb, emb, lr_mul), leaky_relu()])

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = F.normalize(x, dim=1)
        return self.net(x)

class toRGBblock(nn.Module):
    def __init__(self, latent_dim, input_channel, upsample, rgba = False):
        super().__init__()
        self.input_channel = input_channel
        self.to_style      = nn.Linear(latent_dim, input_channel)

        out_filters = 3 if not rgba else 4
        self.conv   = Conv2dModulated(input_channel, out_filters, 1, demod=False)

        self.upsample = nn.Sequential(
            nn.Upsample(scale_factor = 2, mode='bilinear', align_corners=False),
            Blur()
        ) if upsample else None

    def forward(self, x, prev_rgb, istyle):
        b, c, h, w = x.shape
        style      = self.to_style(istyle)
        x          = self.conv(x, style)

        if exists(prev_rgb):
            x = x + prev_rgb

        if exists(self.upsample):
            x = self.upsample(x)

        return x

class Conv2dModulated(nn.Module):
    '''
    Convolution with weights modulation and demodulation.
    '''
    def __init__(
            self,
            in_chan,
            out_chan,
            kernel,
            demod    = True,
            stride   = 1,
            dilation = 1,
            eps      = 1e-8,
            **kwargs
        ):
        super().__init__()
        self.filters  = out_chan
        self.demod    = demod
        self.kernel   = kernel
        self.stride   = stride
        self.dilation = dilation
        self.weight   = nn.Parameter(torch.randn((out_chan, in_chan, kernel, kernel)))
        self.eps      = eps

        nn.init.kaiming_normal_(
            self.weight,
            a=0,
            mode='fan_in',
            nonlinearity='leaky_relu',
        )

    def _get_same_padding(self, size, kernel, dilation, stride):
        return ((size - 1) * (stride - 1) + dilation * (kernel - 1)) // 2

    def forward(self, x, y):
        b, c, h, w = x.shape

        w1      = y[:, None, :, None, None]
        w2      = self.weight[None, :, :, :, :]
        weights = w2 * (w1 + 1)

        if self.demod:
            d   = torch.rsqrt(
                (weights ** 2).sum(dim=(2, 3, 4), keepdim=True) + self.eps
            )
            weights = weights * d

        x = x.reshape(1, -1, h, w)

        _, _, *ws = weights.shape
        weights = weights.reshape(b * self.filters, *ws)

        padding = self._get_same_padding(h, self.kernel, self.dilation, self.stride)
        x = F.conv2d(x, weights, padding=padding, groups=b)

        x = x.reshape(-1, self.filters, h, w)
        return x

### Generator

In [ ]:
class GeneratorBlock(nn.Module):
    '''
    The generator building block.
    '''
    def __init__(
            self,
            latent_dim,
            input_channels,
            filters,
            upsample     = True,
            upsample_rgb = True,
            rgba         = False,
        ):
        super().__init__()
        self.upsample = nn.Upsample(
            scale_factor  = 2,
            mode          = 'bilinear',
            align_corners = False
        ) if upsample else None

        self.to_style1 = nn.Linear(latent_dim,     input_channels)
        self.to_noise1 = nn.Linear(1,              filters)
        self.conv1     = Conv2dModulated(input_channels, filters, 3)

        self.to_style2 = nn.Linear(latent_dim,     filters)
        self.to_noise2 = nn.Linear(1,              filters)
        self.conv2     = Conv2dModulated(filters,  filters, 3)

        self.activation = leaky_relu()
        self.to_rgb     = toRGBblock(
            latent_dim,
            filters,
            upsample_rgb,
            rgba,
            )

    def forward(self, x, prev_rgb, istyle, inoise):
        if exists(self.upsample):
            x = self.upsample(x)

        inoise = inoise[:, :x.shape[2], :x.shape[3], :]
        noise1 = self.to_noise1(inoise).permute((0, 3, 2, 1))
        noise2 = self.to_noise2(inoise).permute((0, 3, 2, 1))

        style1 = self.to_style1(istyle)
        x      = self.conv1(x, style1)
        x      = self.activation(x + noise1)

        style2 = self.to_style2(istyle)
        x      = self.conv2(x, style2)
        x      = self.activation(x + noise2)

        rgb    = self.to_rgb(x, prev_rgb, istyle)

        return x, rgb

In [ ]:
class Generator(nn.Module):
    '''
    The generator network.
    '''
    def __init__(
            self,
            image_size,
            latent_dim,
            filters_scaling_factor = 16,
            transparent            = False,
            attn_layers            = [],
            initial_constant       = True,
            featuremaps_max        = 512,
        ):
        super().__init__()
        self.image_size = image_size
        self.latent_dim = latent_dim
        self.num_layers = int(log2(image_size) - 1)

        filters         = [filters_scaling_factor * (2 ** (i + 1)) \
                          for i in range(self.num_layers)][::-1]

        set_fmap_max    = partial(min, featuremaps_max)
        filters         = list(map(set_fmap_max, filters))
        init_channels   = filters[0]
        filters         = [init_channels, *filters]

        in_out_pairs    = zip(filters[:-1], filters[1:])
        self.initial_constant = initial_constant

        if initial_constant:
            self.initial_block    = nn.Parameter(torch.randn((1, init_channels, 4, 4)))
        else:
            self.to_initial_block = nn.ConvTranspose2d(latent_dim, init_channels, 4, 1, 0, bias=False)

        self.initial_conv = nn.Conv2d(filters[0], filters[0], 3, padding=1)
        self.blocks       = nn.ModuleList([])
        self.attns        = nn.ModuleList([])

        for ind, (in_chan, out_chan) in enumerate(in_out_pairs):
            not_first = ind != 0
            not_last  = ind != (self.num_layers - 1)
            num_layer = self.num_layers - ind

            attn_fn   = attn_and_ff(in_chan) if num_layer in attn_layers else None

            self.attns.append(attn_fn)

            block = GeneratorBlock(
                latent_dim,
                in_chan,
                out_chan,
                upsample     = not_first,
                upsample_rgb = not_last,
                rgba         = transparent
            )
            self.blocks.append(block)

    def forward(self, styles, input_noise):
        batch_size = styles.shape[0]
        image_size = self.image_size

        if self.initial_constant:
            x = self.initial_block.expand(batch_size, -1, -1, -1)
        else:
            avg_style = styles.mean(dim=1)[:, :, None, None]
            x         = self.to_initial_block(avg_style)
 
        rgb    = None
        styles = styles.transpose(0, 1)
        x      = self.initial_conv(x)

        for style, block, attn in zip(styles, self.blocks, self.attns):
            if exists(attn):
                x = attn(x)
            x, rgb = block(x, rgb, style, input_noise)

        return rgb

### Discriminator

In [ ]:
class DiscriminatorBlock(nn.Module):
    '''
    The discriminator building block.
    '''
    def __init__(self, input_channels, filters, downsample=True):
        super().__init__()
        self.conv_res = nn.Conv2d(
            input_channels,
            filters,
            1,
            stride = (2 if downsample else 1),
        )

        self.net = nn.Sequential(
            nn.Conv2d(input_channels, filters, 3, padding=1),
            leaky_relu(),
            nn.Conv2d(filters, filters, 3, padding=1),
            leaky_relu()
        )

        self.downsample = nn.Sequential(
            Blur(),
            nn.Conv2d(filters, filters, 3, padding = 1, stride = 2)
        ) if downsample else None

    def forward(self, x):
        res = self.conv_res(x)
        x   = self.net(x)
        if exists(self.downsample):
            x = self.downsample(x)
        x = (x + res) * (1 / sqrt(2))
        return x

In [ ]:
class Discriminator(nn.Module):
    '''
    The discriminator network.
    '''
    def __init__(
            self,
            image_size,
            filters_scaling_factor = 16,
            vq_disc_layers         = [],
            vq_codebook_size       = 256,
            attn_layers            = [],
            transparent            = False,
            featuremaps_max        = 512,
        ):
        super().__init__()
        num_layers       = int(log2(image_size) - 1)
        num_init_filters = 3 if not transparent else 4

        blocks         = []
        filters        = [num_init_filters] + [(filters_scaling_factor * 4) * (2 ** i) \
                         for i in range(num_layers + 1)]

        set_fmap_max    = partial(min, featuremaps_max)
        filters         = list(map(set_fmap_max, filters))
        chan_in_out     = list(zip(filters[:-1], filters[1:]))

        blocks          = []
        attn_blocks     = []
        quantize_blocks = []

        for ind, (in_chan, out_chan) in enumerate(chan_in_out):
            num_layer   = ind + 1
            is_not_last = ind != (len(chan_in_out) - 1)

            block = DiscriminatorBlock(
                in_chan,
                out_chan,
                downsample = is_not_last,
            )
            blocks.append(block)

            attn_fn = attn_and_ff(out_chan) if num_layer in attn_layers else None

            attn_blocks.append(attn_fn)
    
            quantize_fn = PermuteToFrom(
                VectorQuantize(out_chan, vq_codebook_size)
            ) if num_layer in vq_disc_layers else None
            quantize_blocks.append(quantize_fn)

        self.blocks          = nn.ModuleList(blocks)
        self.attn_blocks     = nn.ModuleList(attn_blocks)
        self.quantize_blocks = nn.ModuleList(quantize_blocks)

        chan_last       = filters[-1]
        latent_dim      = 2 * 2 * chan_last

        self.final_conv = nn.Conv2d(chan_last, chan_last, 3, padding=1)
        self.flatten    = Flatten()
        self.to_logit   = nn.Linear(latent_dim, 1)

    def forward(self, x):
        b, *_ = x.shape

        quantize_loss = torch.zeros(1).to(x)

        for (block, attn_block, q_block) in zip(self.blocks, self.attn_blocks, self.quantize_blocks):
            x = block(x)

            if exists(attn_block):
                x = attn_block(x)

            if exists(q_block):
                x, loss        = q_block(x)
                quantize_loss += loss

        x = self.final_conv(x)
        x = self.flatten(x)
        x = self.to_logit(x)

        return x.squeeze(), quantize_loss

## StyleGAN2 model

In [ ]:
class StyleGAN2(nn.Module):

    def __init__(
            self,
            image_size,
            latent_dim             = 512,
            featuremaps_max        = 512,
            style_depth            = 8,
            filters_scaling_factor = 16,
            transparent            = False,
            fp16                   = False,
            contrastive_regulation = False,
            steps                  = 1,
            lr                     = 1e-4,
            lr_disc_mult           = 2,
            vq_disc_layers         = [],
            vq_codebook_size       = 256,
            attn_layers            = [],
            initial_constant       = True,
            lr_map_network         = 0.1,
            rank                   = 0,
        ):

        super().__init__()
        self.lr          = lr
        self.steps       = steps
        self.ema_updater = EMA(0.995)

        # Style mapping network
        self.S = StyleMapping(
            latent_dim,
            style_depth,
            lr_mul = lr_map_network,
        )
        # Generator network
        self.G = Generator(
            image_size,
            latent_dim,
            filters_scaling_factor,
            transparent      = transparent,
            attn_layers      = attn_layers,
            initial_constant = initial_constant,
            featuremaps_max  = featuremaps_max,
        )
        # Discriminator network
        self.D = Discriminator(
            image_size,
            filters_scaling_factor,
            vq_disc_layers   = vq_disc_layers,
            vq_codebook_size = vq_codebook_size,
            attn_layers      = attn_layers,
            transparent      = transparent,
            featuremaps_max  = featuremaps_max,
        )

        # Style mapping network obtained with exponential moving average
        self.SE = StyleMapping(
            latent_dim,
            style_depth,
            lr_mul = lr_map_network,
        )

        # Generator network obtained with exponential moving average
        self.GE = Generator(
            image_size,
            latent_dim,
            filters_scaling_factor,
            transparent      = transparent,
            attn_layers      = attn_layers,
            initial_constant = initial_constant,
        )

        self.D_cl = None

        # Contrastive regularization term for the discriminator loss
        if contrastive_regulation:
            from contrastive_learner import ContrastiveLearner
            assert not transparent, \
                'contrastive loss regularization does not work with transparent images yet'
            self.D_cl = ContrastiveLearner(self.D, image_size, hidden_layer='flatten')

        # Wrapper for augmenting all images going into the discriminator
        self.D_aug = AugWrapper(self.D, image_size)

        # Turn off gradients calculation in exponential moving average models
        set_requires_grad(self.SE, False)
        set_requires_grad(self.GE, False)

        # Select the optimizers
        generator_params = list(self.G.parameters()) + list(self.S.parameters())
        self.G_opt = optim.Adam(
            generator_params,
            lr    = self.lr,
            betas = (0.5, 0.9),
        )
        self.D_opt = optim.Adam(
            self.D.parameters(),
            lr    = self.lr * lr_disc_mult,
            betas = (0.5, 0.9),
        )

        # Initialize the weights
        self._init_weights()
        self.reset_parameter_averaging()

        self.cuda(rank)

        # Start APEX mixed precision
        self.fp16 = fp16
        if fp16:
            (self.S, self.G, self.D, self.SE, self.GE), (self.G_opt, self.D_opt) = \
                amp.initialize(
                    [self.S, self.G, self.D, self.SE, self.GE],
                    [self.G_opt, self.D_opt],
                    opt_level='O1',
                    num_losses=3,
                )

    def _init_weights(self):
        for m in self.modules():
            if type(m) in {nn.Conv2d, nn.Linear}:
                nn.init.kaiming_normal_(
                    m.weight,
                    a            = 0,
                    mode         = 'fan_in',
                    nonlinearity = 'leaky_relu',
                )

        for block in self.G.blocks:
            nn.init.zeros_(block.to_noise1.weight)
            nn.init.zeros_(block.to_noise2.weight)
            nn.init.zeros_(block.to_noise1.bias)
            nn.init.zeros_(block.to_noise2.bias)

    def EMA(self):
        def update_moving_average(ma_model, current_model):
            for current_params, ma_params in zip(current_model.parameters(), ma_model.parameters()):
                old_weight, up_weight = ma_params.data, current_params.data
                ma_params.data        = self.ema_updater.update_average(old_weight, up_weight)

        update_moving_average(self.SE, self.S)
        update_moving_average(self.GE, self.G)

    def reset_parameter_averaging(self):
        self.SE.load_state_dict(self.S.state_dict())
        self.GE.load_state_dict(self.G.state_dict())

    def forward(self, x):
        return x

## Trainer class

In [ ]:
class Trainer():

    def __init__(
        self,
        epoch,
        *args,
        **config
        ):

        self.GAN_params                = [args, config]
        self.GAN                       = None
        self.epoch                     = epoch

        self.name                      = config['experiment_name']
        self.dataset_name              = config['dataset_name']
        self.data_path                 = config['data_path']
        root_dir                       = Path(config['base_dir'])
        self.base_dir                  = root_dir
        self.results_dir               = root_dir / config['results_dir'] / self.name
        self.models_dir                = root_dir / config['models_dir'] / self.name
        self.fid_dir                   = root_dir / 'fid' / self.name
        self.config_path               = self.models_dir / 'config.json'

        assert log2(config['image_size']).is_integer(), \
            'Image size must be a power of 2 (64, 128, 256, 512, 1024)'
        self.image_size                = config['image_size']
        self.channels                  = config['channels']
        self.latent_dim                = config['latent_dim']
        self.crop_size                 = config['crop_size']
        self.filters_scaling_factor    = config['filters_scaling_factor']
        self.featuremaps_max           = config['featuremaps_max']
        self.transparent               = config['transparent']

        self.vq_disc_layers            = cast_list(config['vq_disc_layers'])
        self.vq_codebook_size          = config['vq_codebook_size']
        self.has_fq                    = len(self.vq_disc_layers) > 0

        self.attn_layers               = cast_list(config['attn_layers'])
        self.initial_constant          = config['initial_constant']

        self.aug_prob                  = config['aug_prob']
        self.aug_types                 = config['aug_types']

        self.lr                        = config['lr']
        self.lr_map_network            = config['lr_map_network']
        self.lr_disc_mult              = config['lr_disc_mult']
        self.relative_disc_losses      = config['relative_disc_losses']
        self.batch_size                = config['batch_size']
        self.num_workers               = config['num_workers']
        self.mixed_prob                = config['mixed_prob']

        self.grid_size                 = config['grid_size']
        self.evaluate_every_steps      = config['evaluate_every_steps']
        self.save_every_steps          = None
        self.log_interval              = config['log_interval']
        self.pl_log_interval           = int(config["log_interval"] / 16)
        self.step                      = 0

        self.av                        = None
        self.truncation_psi            = config['truncation_psi']

        self.path_length_regulation    = config['path_length_regulation']
        self.pl_every_steps            = config['pl_every_steps']
        self.pl_mean                   = None

        self.gp_every_steps            = config['gp_every_steps']

        self.gradient_accumulate_every = config['gradient_accumulate_every']

        assert not config['fp16'] or config['fp16'] and APEX_AVAILABLE, \
            'Apex is not available for you to use mixed precision training'
        self.fp16                      = config['fp16']

        self.contrastive_regulation    = config['contrastive_regulation']

        self.d_loss                    = 0
        self.g_loss                    = 0
        self.q_loss                    = None
        self.last_gp_loss              = None
        self.last_cr_loss              = None
        self.last_fid                  = 0

        self.pl_length_ma              = EMA(0.99)
        self.init_folders()

        self.loader                    = None
        self.dataset_aug_prob          = config['dataset_aug_prob']

        self.calculate_fid_interval    = config['calculate_fid_interval']
        self.calculate_fid_num_images  = config['calculate_fid_num_images']
        self.clear_fid_cache           = config['clear_fid_cache']

        self.generator_top_k           = config['generator_top_k']
        self.generator_top_k_gamma     = config['generator_top_k_gamma']
        self.generator_top_k_frac      = config['generator_top_k_frac']

        self.dual_contrastive_loss     = config['dual_contrastive_loss']  

        assert not (config['is_ddp'] and config['contrastive_regulation']), \
            'Contrastive loss regularization does not work well with multi GPUs yet'
        self.is_ddp                    = config['is_ddp']
        self.is_main                   = (config['rank'] == 0)
        self.rank                      = config['rank']
        self.world_size                = config['world_size']

        # Create an empty dictionary to store the training results
        self.results = {
            'g_loss':              [],
            'd_loss':              [],
            'gp':                  [],
            'plp':                 [],
            'epoch_training_time': [],
        }

    # Function 'image_extension' is treated as an attribute of the enclosing class
    # and as so can be used as:
    # 'self.image_extension = ...'
    # '... = self.image_extension'
    @property
    def image_extension(self):
        return 'jpg' if not self.transparent else 'png'

    # Function 'checkpoint_num' is treated as an attribute of the enclosing class
    # and as so can be used as:
    # 'self.checkpoint_num = ...'
    # '... = self.checkpoint_num'
    @property
    def checkpoint_num(self):
        return floor((self.step+1) // self.save_every_steps)

    # Function 'hparams' is treated as an attribute of the enclosing class
    # and as so can be used as:
    # 'self.hparams = ...'
    # '... = self.hparams'
    @property
    def hparams(self):
        return {'image_size': self.image_size, 'filters_scaling_factor': self.filters_scaling_factor}

    def init_GAN(self):
        '''
        Creates a new instance of the StyleGAN2 class.
        '''
        args, config = self.GAN_params
        self.GAN = StyleGAN2(
            latent_dim             = self.latent_dim,
            lr                     = self.lr,
            lr_map_network         = self.lr_map_network,
            lr_disc_mult           = self.lr_disc_mult,
            image_size             = self.image_size,
            filters_scaling_factor = self.filters_scaling_factor,
            featuremaps_max        = self.featuremaps_max,
            transparent            = self.transparent,
            vq_disc_layers         = self.vq_disc_layers,
            vq_codebook_size       = self.vq_codebook_size,
            attn_layers            = self.attn_layers,
            fp16                   = self.fp16,
            contrastive_regulation = self.contrastive_regulation,
            initial_constant       = self.initial_constant,
            rank                   = self.rank,
            *args,
        )

        if self.is_ddp:
            ddp_config     = {'device_ids': [self.rank]}
            self.S_ddp     = DDP(self.GAN.S,     **ddp_config)
            self.G_ddp     = DDP(self.GAN.G,     **ddp_config)
            self.D_ddp     = DDP(self.GAN.D,     **ddp_config)
            self.D_aug_ddp = DDP(self.GAN.D_aug, **ddp_config)

    def write_config(self):
        '''
        Writes the configuration to a JSON file.
        '''
        self.config_path.write_text(json.dumps(self.config()))

    def load_config(self):
        '''
        Loads configuration from file, removes current instance of StyleGAN
        and creates a new instance of the StyleGAN2 class.
        '''
        config                      = self.config() if not self.config_path.exists() \
                                      else json.loads(self.config_path.read_text())
        self.image_size             = config['image_size']
        self.filters_scaling_factor = config['filters_scaling_factor']
        self.transparent            = config['transparent']
        self.vq_disc_layers         = config['vq_disc_layers']
        self.vq_codebook_size       = config['vq_codebook_size']
        self.featuremaps_max        = config.pop('featuremaps_max', 512)
        self.attn_layers            = config.pop('attn_layers', [])
        self.initial_constant       = config.pop('initial_constant', False)
        self.lr_map_network         = config.pop('lr_map_network', 0.1)
        print('[INFO] Deleting current StyleGAN2 model present in the training session <2> ...')
        del self.GAN
        print('[INFO] Creating a new StyleGAN2 model in the training session <2> ...')
        self.init_GAN()

    def config(self):
        '''
        Returns the main hyperparameters of the current Trainer.
        '''
        return {
            'image_size':             self.image_size,
            'filters_scaling_factor': self.filters_scaling_factor,
            'lr_map_network':         self.lr_map_network,
            'transparent':            self.transparent,
            'vq_disc_layers':         self.vq_disc_layers,
            'vq_codebook_size':       self.vq_codebook_size,
            'attn_layers':            self.attn_layers,
            'initial_constant':       self.initial_constant
        }

    def setup_dataset(self):
        '''
        Setup a custom dataset from images in folder and DataLoader to
        retrieve batches of images from that dataset.
        '''
        if self.crop_size != None:
            offset_height = (218 - self.crop_size) // 2
            offset_width  = (178 - self.crop_size) // 2
            crop = lambda x: x[:, offset_height:offset_height + self.crop_size, offset_width:offset_width + self.crop_size]

            train_transform = transforms.Compose(
                [
                    transforms.ToTensor(),
                    transforms.Lambda(crop),
                    transforms.Resize(size=(self.image_size, self.image_size), antialias=True),
                    #transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
                ]
            )
        else:
            train_transform = transforms.Compose(
                [
                    transforms.ToTensor(),
                    transforms.Resize(size=(self.image_size, self.image_size), antialias=True),
                    #transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
                ]
            )

        self.dataset = CustomDataSet(
            root_dir    = self.data_path,
            transform   = train_transform,
            transparent = self.transparent,
            aug_prob    = self.dataset_aug_prob,
        )
        print(f'Dataset number of images: {len(self.dataset)}')

        num_workers = default(
            self.num_workers,
            NUM_CORES if not self.is_ddp else 0,
        )

        sampler = DistributedSampler(
            self.dataset,
            rank         = self.rank,
            num_replicas = self.world_size,
            shuffle      = True,
        ) if self.is_ddp else None

        dataloader = DataLoader(
            self.dataset,
            num_workers  = num_workers,
            batch_size   = ceil(self.batch_size / self.world_size),
            sampler      = sampler,
            shuffle      = not self.is_ddp,
            drop_last    = True,
            pin_memory   = True,
        )
        #self.loader = cycle(dataloader)
        self.loader = dataloader

        # auto set augmentation prob for user if dataset is detected to be low
        num_samples = len(self.dataset)
        if not exists(self.aug_prob) and num_samples < 1e5:
            self.aug_prob = min(0.5, (1e5 - num_samples) * 3e-6)
            print(f'Set automatic augmentation probability to {round(self.aug_prob * 100)}%')

    def check_dataloader(self):
        '''
        Checks if the created dataloader works fine and display a few real images.
        '''
        NR, NC    = 3, 3
        self.setup_dataset()
        imgs      = next(iter(self.loader))
        print(f'Batch of images shape: {imgs.shape}')   # BS, Ch, H, W

        if NR*NC > imgs.shape[0]:
            NR = 2
            if NR*NC > imgs.shape[0]:
                NR = 1
                if NR*NC > imgs.shape[0]:
                    NC = 2

        _, ax = plt.subplots(NR, NC, figsize=(3*NC,3*NR))
        plt.suptitle(
            f'Some real images of {self.dataset_name} dataset',
            fontsize=15,
            fontweight='bold'
        )

        index = 0
        for r in range(NR):
            for c in range(NC):
                index += 1
                if NR==1:
                    #ax[c].imshow(imgs[index].permute(1,2,0)+1)/2)
                    ax[c].imshow(imgs[index].permute(1,2,0))
                else:
                    #ax[r][c].imshow((imgs[index].permute(1,2,0)+1)/2)
                    ax[r][c].imshow(imgs[index].permute(1,2,0))

    def train_step(self):
        '''
        Train StyleGAN during one step/batch.
        '''
        assert exists(self.loader), '[WARN] You must first initialize the data source with .setup_dataset(<folder of images>)'

        if self.GAN is None:
            print('[INFO] Creating a new StyleGAN2 model in the training session ...')
            self.init_GAN()

        # Put the model in training mode
        self.GAN.train()

        # Model setup: loss=0, batch size, image size, latent dimension, #layers
        total_disc_loss = torch.tensor(0.).cuda(self.rank)
        total_gen_loss  = torch.tensor(0.).cuda(self.rank)

        batch_size      = ceil(self.batch_size / self.world_size)
        image_size      = self.GAN.G.image_size
        latent_dim      = self.GAN.G.latent_dim
        num_layers      = self.GAN.G.num_layers

        aug_prob        = self.aug_prob
        aug_types       = self.aug_types
        aug_kwargs      = {'prob': aug_prob, 'types': aug_types}

        # Decide if gradient penalty term is added to the loss: every 4 steps only
        apply_gradient_penalty    = ((self.step+1) % self.gp_every_steps == 0)
        gp_mean_length            = self.log_interval // self.gp_every_steps

        # Decide if path lenth penalty term is added to the loss
        apply_path_penalty        = not self.path_length_regulation and (self.step+1) > 5000 and (self.step+1) % self.pl_every_steps == 0
        apply_cl_reg_to_generated = (self.step+1) > 20000

        S         = self.GAN.S if not self.is_ddp else self.S_ddp
        G         = self.GAN.G if not self.is_ddp else self.G_ddp
        D         = self.GAN.D if not self.is_ddp else self.D_ddp
        D_aug     = self.GAN.D_aug if not self.is_ddp else self.D_aug_ddp

        backwards = partial(loss_backwards, self.fp16)

        # Calculate contrastive regularization term to be added to the discriminator loss
        if exists(self.GAN.D_cl):
            self.GAN.D_opt.zero_grad()

            if apply_cl_reg_to_generated:
                for i in range(self.gradient_accumulate_every):
                    get_latents_fn   = mixed_list if random() < self.mixed_prob else noise_list
                    style            = get_latents_fn(batch_size, num_layers, latent_dim, device=self.rank)
                    noise            = image_noise(batch_size, image_size, device=self.rank)

                    w_space          = latent_to_w(self.GAN.S, style)
                    w_styles         = styles_def_to_tensor(w_space)

                    generated_images = self.GAN.G(w_styles, noise)
                    self.GAN.D_cl(generated_images.clone().detach(), accumulate=True)

            # Repeat 'self.gradient_accumulate_every' times
            for i in range(self.gradient_accumulate_every):

                # Load a batch of images from dataset
                image_batch = next(iter(self.loader)).cuda(self.rank)
                self.GAN.D_cl(image_batch, accumulate=True)

            loss              = self.GAN.D_cl.calculate_loss()
            self.last_cr_loss = loss.clone().detach().item()
            backwards(loss, self.GAN.D_opt, loss_id = 0)

            self.GAN.D_opt.step()

        # Select the losses ..............................................................

        if not self.dual_contrastive_loss:
            D_loss_fn        = hinge_loss
            G_loss_fn        = gen_hinge_loss
            G_requires_reals = False
        else:
            D_loss_fn        = dual_contrastive_loss
            G_loss_fn        = dual_contrastive_loss
            G_requires_reals = True

        # Train the discriminator ........................................................

        avg_pl_length = self.pl_mean
        self.GAN.D_opt.zero_grad()

        for i in gradient_accumulate_contexts(self.gradient_accumulate_every, self.is_ddp, ddps=[D_aug, S, G]):

            get_latents_fn   = mixed_list if random() < self.mixed_prob else noise_list
            style            = get_latents_fn(batch_size, num_layers, latent_dim, device=self.rank)
            noise            = image_noise(batch_size, image_size, device=self.rank)

            w_space          = latent_to_w(S, style)
            w_styles         = styles_def_to_tensor(w_space)

            generated_images = G(w_styles, noise)
            generated_images, (fake_output, fake_q_loss) = D_aug(
                generated_images.clone().detach(),
                return_aug_images   = True,
                input_requires_grad = apply_gradient_penalty,
                detach              = True,
                **aug_kwargs
            )

            image_batch = next(iter(self.loader)).cuda(self.rank)

            if apply_gradient_penalty:
                image_batch.requires_grad_()

            real_output, real_q_loss = D_aug(image_batch, **aug_kwargs)

            real_output_loss = real_output
            fake_output_loss = fake_output

            if self.relative_disc_losses:
                real_output_loss = real_output_loss - fake_output.mean()
                fake_output_loss = fake_output_loss - real_output.mean()

            divergence = D_loss_fn(fake_output_loss, real_output_loss)
            disc_loss  = divergence

            if self.has_fq:
                quantize_loss = (fake_q_loss + real_q_loss).mean()
                self.q_loss   = float(quantize_loss.detach().item())

                disc_loss = disc_loss + quantize_loss

            if apply_gradient_penalty:
                gp = gradient_penalty(image_batch, real_output) + \
                    gradient_penalty(generated_images, fake_output)
                self.last_gp_loss = gp.clone().detach().item()
                disc_loss         = disc_loss + gp

            disc_loss = disc_loss / self.gradient_accumulate_every
            disc_loss.register_hook(raise_if_nan)
            backwards(disc_loss, self.GAN.D_opt, loss_id = 1)

            total_disc_loss += divergence.detach().item() / self.gradient_accumulate_every

        self.d_loss = float(total_disc_loss)

        self.GAN.D_opt.step()

        # Train the generator ..........................................................

        self.GAN.G_opt.zero_grad()

        for i in gradient_accumulate_contexts(self.gradient_accumulate_every, self.is_ddp, ddps=[S, G, D_aug]):

            style = get_latents_fn(batch_size, num_layers, latent_dim, device=self.rank)
            noise = image_noise(batch_size, image_size, device=self.rank)

            w_space          = latent_to_w(S, style)
            w_styles         = styles_def_to_tensor(w_space)

            generated_images = G(w_styles, noise)
            fake_output, _   = D_aug(generated_images, **aug_kwargs)
            fake_output_loss = fake_output

            real_output = None
            if G_requires_reals:
                image_batch    = next(iter(self.loader)).cuda(self.rank)
                real_output, _ = D_aug(image_batch, detach = True, **aug_kwargs)
                real_output    = real_output.detach()

            # Update the Generator using gradients from the top-k most realistic samples
 
            if self.generator_top_k:
                # Obtain the current epoch

                #topk_epoch = (self.step * batch_size * self.gradient_accumulate_every) / len(self.dataset)
                topk_epoch  = self.epoch
                
                # Calculate the current fraction of batch samples whose gradient 
                # will be considered by the top-k technique to update the generator:

                # current_fraction = maximum(gamma^epoch, fraction) 
                k_frac      = max(self.generator_top_k_gamma ** topk_epoch, self.generator_top_k_frac)

                # current_k = current_fraction * batch_size
                k           = ceil(batch_size * k_frac)

                if k != batch_size:
                    fake_output_loss, _ = fake_output_loss.topk(k=k, largest=False)

            loss     = G_loss_fn(fake_output_loss, real_output)
            gen_loss = loss

            if apply_path_penalty:
                pl_lengths    = calc_pl_lengths(w_styles, generated_images)
                avg_pl_length = np.mean(pl_lengths.detach().cpu().numpy())

                if not is_empty(self.pl_mean):
                    pl_loss = ((pl_lengths - self.pl_mean) ** 2).mean()
                    if not torch.isnan(pl_loss):
                        gen_loss = gen_loss + pl_loss

            gen_loss = gen_loss / self.gradient_accumulate_every
            gen_loss.register_hook(raise_if_nan)
            backwards(gen_loss, self.GAN.G_opt, loss_id = 2)

            total_gen_loss += loss.detach().item() / self.gradient_accumulate_every

        self.g_loss = float(total_gen_loss)

        self.GAN.G_opt.step()

        # Calculate the moving averages ..................................................

        compute_pl_mean = (apply_path_penalty and not np.isnan(avg_pl_length))
        if compute_pl_mean:
            self.pl_mean = self.pl_length_ma.update_average(self.pl_mean, avg_pl_length)

        if self.is_main and (self.step+1) % 10 == 0 and (self.step+1) > 20000:
            self.GAN.EMA()

        if self.is_main and (self.step+1) <= 25000 and (self.step+1) % 1000 == 2:
            self.GAN.reset_parameter_averaging()

        # Protect training from NaN errors ..............................................

        if any(torch.isnan(l) for l in (total_gen_loss, total_disc_loss)):
            print(f'NaN detected for generator or discriminator. Loading from checkpoint {self.checkpoint_num}')
            if self.checkpoint_num > 0:
                _ = self.load(checkpoint_num=self.checkpoint_num)
            else:
                print('There is no checkpoint to load yet. You have to start training from the beginning')
            raise NanException

        # Save and log results ..........................................................

        if self.is_main:

            if (self.step+1) % self.save_every_steps == 0:
                self.save(self.checkpoint_num, config)

            if (self.step+1) % self.evaluate_every_steps == 0 or ((self.step+1) % 100 == 0 and (self.step+1) < 2500):
                self.evaluate(self.epoch, self.step)

            if exists(self.calculate_fid_interval) and (self.step+1) % self.calculate_fid_interval == 0 and self.step != 0:
                num_batches   = ceil(self.calculate_fid_num_images / self.batch_size)
                try:
                    fid           = self.calculate_fid(num_batches)
                    self.last_fid = fid
                    with open(str(self.results_dir / f'fid_scores.txt'), 'a') as f: 
                        f.write(f'{self.step+1},{fid}\n')
                except ValueError:
                    print("[ERROR] calculate_fid() got an error during calculations")

            # Save the results in a dictionary ......................................

            self.results["g_loss"].append(self.g_loss)
            self.results["d_loss"].append(self.d_loss)
            if apply_gradient_penalty:
                self.results["gp"].append(self.last_gp_loss)
            if compute_pl_mean==True and (self.step+1) % 16 == 0: 
                self.results["plp"].append(self.pl_mean)
            #if exists(self.calculate_fid_interval):
            #    self.results["fid"].append(self.last_fid)

            # Save progress metrics to W&B ..........................................

            if (self.step+1) % self.log_interval == 0:

                mean_g_loss      = np.mean(self.results["g_loss"][-self.log_interval:])
                mean_d_loss      = np.mean(self.results["d_loss"][-self.log_interval:])
                mean_gp          = np.mean(self.results["gp"][-gp_mean_length:])
                if compute_pl_mean==True:
                    mean_plp     = np.mean(self.results["plp"][-self.pl_log_interval:])
                else:
                    mean_plp     = None

                self.track_WandB(
                    compute_pl_mean,
                    mean_g_loss,
                    mean_d_loss,
                    mean_gp,
                    mean_plp,
                )

        self.av     = None

    @torch.no_grad()
    def evaluate(self, epoch, step=0, postfix=""):
        '''
        Generate a grid of images in different scenarios:
        - with the regular style mapping and generator networks,
        - with the EMA style mapping and the EMA generator networks,
        - with the EMA style mapping network, the EMA generator and style mixing.
        '''
        # Put the model in evaluation mode
        self.GAN.eval()

        ext        = self.image_extension # get the extension of the file to be created
        num_rows   = self.grid_size       # Size of the grif of images to create

        latent_dim = self.GAN.G.latent_dim
        image_size = self.GAN.G.image_size
        num_layers = self.GAN.G.num_layers

        # Latents: generate 'num_rows' x 'num_rows' normally distributed random vectors 
        #          with size 'latent_dim' for each layer.

        latents = noise_list(num_rows ** 2, num_layers, latent_dim, device=self.rank)

        # Noise to be added to the feature maps: generate 'num_rows' x 'num_rows' uniformly
        #    distributed random noise matrices, each with size 'image_size' x 'image_size'.
        n       = image_noise(num_rows ** 2, image_size, device=self.rank)

        # Generate a grid of 'num_rows' x 'num_rows' images using the style mapping network 
        # self.GAN.S and the generator self.GAN.G

        generated_images = self.generate_truncated(
            self.GAN.S,
            self.GAN.G,
            latents,
            n,
            truncation_psi = self.truncation_psi,
        )
        # Save the grid of generated images to a file in the results folder.
        save_image(
            generated_images,
            str(self.results_dir / f'generated_images_epoch{str(epoch+1).zfill(3)}_step{str(step+1).zfill(6)}.{ext}'),
            nrow=num_rows,
        )

        # Generate a grid of 'num_rows' x 'num_rows' images using the style EMA mapping network
        # self.GAN.SE and the EMA generator self.GAN.GE

        generated_images = self.generate_truncated(
            self.GAN.SE,
            self.GAN.GE,
            latents,
            n,
            truncation_psi = self.truncation_psi,
        )
        # Save the grid of generated images to a file in the results folder.
        save_image(
            generated_images,
            str(self.results_dir / f'generated_images_epoch{str(epoch+1).zfill(3)}_step{str(step+1).zfill(6)}_ema.{ext}'),
            nrow=num_rows,
        )

        # Explore style mixing

        def tile(a, dim, n_tile):
            init_dim        = a.size(dim)
            repeat_idx      = [1] * a.dim()
            repeat_idx[dim] = n_tile
            a               = a.repeat(*(repeat_idx))
            order_index     = torch.LongTensor(
                np.concatenate(
                    [init_dim * np.arange(n_tile) + i for i in range(init_dim)]
                )
            ).cuda(self.rank)
            return torch.index_select(a, dim, order_index)

        # Get a noise tensor with size [nun_rows x latent_dim] filled with random values from N(0,1)
        nn   = noise(num_rows, latent_dim, device=self.rank)

        # Replicate each row of 'nn' a number of times equal 'num_rows' -> tmp1
        tmp1 = tile(nn, 0, num_rows)
        # Replicate 'nn' a number of times equal 'num_rows' -> tmp2
        tmp2 = nn.repeat(num_rows, 1)

        # Create two tuples: (tmp1, num_layers/2) (tmp2, num_layers/2-num_layers/2)
        tt            = int(num_layers / 2)
        mixed_latents = [(tmp1, tt), (tmp2, num_layers - tt)]

        # Generate a grid of num_rows x num_rows images using the style EMA mapping network
        # self.GAN.SE and the EMA generator self.GAN.GE and aplying style mixing.
        # Half generator layers receives a set o random vectors 'tmp1' and the other
        # half receives a a different set o random vectors 'tmp2'.

        generated_images = self.generate_truncated(
            self.GAN.SE,
            self.GAN.GE,
            mixed_latents,
            n,
            truncation_psi = self.truncation_psi,
        )
        # Save the grid of generated images to a file in the results folder.
        save_image(
            generated_images,
            str(self.results_dir / f'generated_images_epoch{str(epoch+1).zfill(3)}_step{str(step+1).zfill(6)}_style_mixing.{ext}'),
            nrow=num_rows,
        )

    @torch.no_grad()
    def calculate_fid(self, num_batches):
        '''
        Calculate the FID between 'num_batches' batches of real images and generated images.
        '''
        torch.cuda.empty_cache()

        real_path = self.fid_dir / 'real'
        fake_path = self.fid_dir / 'fake'

        # Remove any existing real images used for fid calculation and recreate directories

        if not real_path.exists() or self.clear_fid_cache:
            rmtree(real_path, ignore_errors=True)
            os.makedirs(real_path)

            for batch_num in tqdm(range(num_batches), desc='calculating FID - saving real images'):
                real_batch = next(iter(self.loader))
                for k, image in enumerate(real_batch.unbind(0)):
                    filename = str(k + batch_num * self.batch_size)
                    save_image(image, str(real_path / f'{filename}.png'))

        # Generate a bunch of fake images and place them in folder 'fid/exp_name/fake'

        rmtree(fake_path, ignore_errors=True)
        os.makedirs(fake_path)

        self.GAN.eval()
        ext        = self.image_extension

        latent_dim = self.GAN.G.latent_dim
        image_size = self.GAN.G.image_size
        num_layers = self.GAN.G.num_layers

        for batch_num in tqdm(range(num_batches), desc='calculating FID - saving generated images'):

            # Latents: generate 'batch_size' normally distributed random vectors
            #          with size 'latent_dim' for each layer

            latents = noise_list(self.batch_size, num_layers, latent_dim, device=self.rank)

            # Noise to be added to the feature maps: generate 'batch_size' uniformly
            #    distributed random noise matrices, each with size 'image_size' x 'image_size'.

            noise   = image_noise(self.batch_size, image_size, device=self.rank)

            # Generate 'batch_size' images using the style EMA mapping network
            # self.GAN.SE and the EMA generator self.GAN.GE

            generated_images = self.generate_truncated(
                self.GAN.SE,
                self.GAN.GE,
                latents,
                noise,
                truncation_psi = self.truncation_psi,
            )

            # Save the generated images to files
            for j, image in enumerate(generated_images.unbind(0)):
                save_image(
                    image,
                    str(fake_path / f'{str(j + batch_num * self.batch_size)}-ema.{ext}'),
                )

        # Calculate the FID between the real and generated images.
        return fid_score.calculate_fid_given_paths(
            [str(real_path), str(fake_path)],
            256,
            noise.device,
            2048,
            )

    @torch.no_grad()
    def truncate_style(self, tensor, S = None, truncation_psi = 0.75):
        '''
        Truncate styles around the average style using the truncation factor psi.
        '''
        S          = default(S, self.GAN.S)
        batch_size = self.batch_size
        latent_dim = self.GAN.G.latent_dim

        # Obtain a kind of average style calculated batch by batch and truncate
        # the style using the average style and the truncation factor psi:
        #    w_trunc = w_avg + (w - w_avg) * psi

        if not exists(self.av):
            z       = noise(2000, latent_dim, device=self.rank)
            samples = evaluate_in_chunks(batch_size, S, z).cpu().numpy()
            self.av = np.mean(samples, axis = 0)
            self.av = np.expand_dims(self.av, axis = 0)

        av_torch    = torch.from_numpy(self.av).cuda(self.rank)
        tensor      = truncation_psi * (tensor - av_torch) + av_torch
        return tensor

    @torch.no_grad()
    def truncate_style_defs(self, w, S = None, truncation_psi = 0.75):
        '''
        For each generator block, obtain truncated styles around the average style
        using the truncation factor psi.
        '''
        w_space = []
        for tensor, num_layers in w:
            tensor = self.truncate_style(tensor, S = S, truncation_psi = truncation_psi)
            w_space.append((tensor, num_layers))
        return w_space

    @torch.no_grad()
    def generate_truncated(self, S, G, style, noise, truncation_psi = 0.75, grid_size = 5):
        '''
        Generate a grid of images using style sampling truncation.
        '''
        w                = map(lambda t: (S(t[0]), t[1]), style)
        w_truncated      = self.truncate_style_defs(w, S = S, truncation_psi = truncation_psi)
        w_styles         = styles_def_to_tensor(w_truncated)
        generated_images = evaluate_in_chunks(self.batch_size, G, w_styles, noise)
        return generated_images.clamp_(0., 1.)

    @torch.no_grad()
    def generate_interpolation(self, num = 0, grid_size = 8, trunc = 1.0, num_steps = 100, save_frames = False):
        '''
        Generate a sequence of (grids of) images, for a sequence of latents
        obtained by spherical interpolation between two latents (low and high.
        '''
        self.GAN.eval()
        ext          = self.image_extension
        num_rows     = grid_size

        latent_dim   = self.GAN.G.latent_dim
        image_size   = self.GAN.G.image_size
        num_layers   = self.GAN.G.num_layers

        # Latents low: generate 'num_rows x num_rows' normally distributed random vectors
        #              with size 'latent_dim' for each layer

        latents_low  = noise(num_rows ** 2, latent_dim, device=self.rank)

        # Latents high: generate another set of 'num_rows x num_rows' normally distributed
        #               random vectors with size 'latent_dim' for each layer

        latents_high = noise(num_rows ** 2, latent_dim, device=self.rank)

        # Noise to be added to the feature maps: generate 'num_rows x num_rows' uniformly
        #    distributed random noise matrices, each with size 'image_size' x 'image_size'.
        n            = image_noise(num_rows ** 2, image_size, device=self.rank)

        # Generate 'num_steps' values equally spaced in the interval [0:8]
        ratios       = torch.linspace(0., 8., num_steps)

        frames = []
        for ratio in tqdm(ratios):

            # Apply spherical interpolation to obtain an interplated latent set located between
            # the latent low and the latent high sets.
            interp_latents   = slerp(ratio, latents_low, latents_high)
            latents          = [(interp_latents, num_layers)]

            # Generate 'num_rows x num_rows' images using the style EMA mapping network
            # self.GAN.SE and the EMA generator self.GAN.GE

            generated_images = self.generate_truncated(
                self.GAN.SE,
                self.GAN.GE,
                latents,
                n,
                truncation_psi = self.truncation_psi,
            )
            # Make a grid with the generated images
            images_grid      = make_grid(generated_images, nrow = num_rows)
            pil_image        = transforms.ToPILImage()(images_grid.cpu())

            if self.transparent:
                background = Image.new("RGBA", pil_image.size, (255, 255, 255))
                pil_image  = Image.alpha_composite(background, pil_image)

            # Add the grid of images to 'frames' list
            frames.append(pil_image)

        # Save the list/sequence of images to a GIF file.
        frames[0].save(
            str(self.results_dir / self.name / f'{str(num)}.gif'),
            save_all      = True,
            append_images = frames[1:],
            duration      = 80,
            loop          = 0,
            optimize      = True,
        )

        # Save each image in 'frames' to a separate file.
        if save_frames:
            folder_path = (self.results_dir / self.name / f'{str(num)}')
            folder_path.mkdir(parents=True, exist_ok=True)
            for ind, frame in enumerate(frames):
                frame.save(str(folder_path / f'{str(ind)}.{ext}'))

    def print_metrics(self, tqdm_loop):
        '''
        Print progress metrics.
        '''
        data = [
            ('Epoch',   self.epoch+1),
            ('step',    self.step+1),
            ('G_loss',  self.g_loss),
            ('D_loss',  self.d_loss),
            ('GP_loss', self.last_gp_loss),
            ('PL_loss', self.pl_mean),
            ('CR_loss', self.last_cr_loss),
            ('Q_loss',  self.q_loss),
            ('FID',     self.last_fid)
        ]

        l_epoch   = data[0]
        log_epoch = f'{l_epoch[0]} : {l_epoch[1]} | '
        l_step    = data[1]
        log_step  = f'{l_step[0]} : {l_step[1]} | '

        data      = [d for d in data[2:] if d[1] is not None]
        log_rest  = ' | '.join(map(lambda n: f'{n[0]}: {n[1]:.3f}', data))

        #print(log_epoch, log_step, log_rest)
        tqdm_loop.set_postfix_str(f"{log_epoch}{log_step}{log_rest}")

    def track_WandB(self, compute_pl_mean, g_loss, d_loss, gp, plp):
        '''
        Send progress metrics to Weights and Biases.
        '''
        try:
            # Log metrics to Weights & Biases ............................
            wb_metrics = {
                "g_loss":   g_loss,
                "d_loss":   d_loss,
                "gp":       gp,
                }
            if compute_pl_mean == True:
                wb_metrics['plp'] = plp
            wb_metrics['epoch'] = self.epoch+1
            wandb.log(wb_metrics)
        except Exception as ex:
            print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

    def model_name(self, checkpoint_num):
        '''
        Defines the name of a checkpoint file.
        '''
        return str(self.models_dir / f'{self.name}_epoch{self.epoch+1}_{checkpoint_num}.pth')

    def init_folders(self):
        '''
        Recreates the results and model folders.
        '''
        self.results_dir.mkdir(parents=True, exist_ok=True)
        self.models_dir.mkdir(parents=True,  exist_ok=True)

    def clear(self):
        '''
        Removes the results folder, the models folder, the FID folder and
        configuration file for current experiment.
        '''
        rmtree(str(self.models_dir), True)
        rmtree(str(self.results_dir), True)
        rmtree(str(self.fid_dir), True)
        if os.path.exists(self.config_path):
            os.remove(self.config_path)
            print(f"JSON configuration file '{self.config_path}' deleted successfully.")
        else:
            print(f"JSON configuration file '{self.config_path}' not found.")
        self.init_folders()

    def save(
            self,
            checkpoint_num,
            config,
        ):
        '''
        Saves to a pth file:
        - the StyleGAN2 model checkpoint (generator, discriminator, mapping network, optimizers)
        - the obtained average results
        - the current epoch ID
        - the training hyperparameters.
        '''
        results_to_save = {
            'version':          __version__,
            'step':             self.step+1,  # add 1 to obtain next step  value when the model is loaded
            'epoch':            self.epoch+1, # add 1 to obtain next epoch value when the model is loaded
            'GAN':              self.GAN.state_dict(),
            'results':          self.results,
            'hyperparameters':  config,
        }

        if self.GAN.fp16:
            results_to_save['amp'] = amp.state_dict()

        torch.save(
            results_to_save,
            self.model_name(checkpoint_num),
        )

        self.write_config()

    def load(self, load_model_from=None, checkpoint_num=None):
        '''
        Given a file name or a checkpoint number, loads from a pth file:
        - the StyleGAN2 model  (generator, discriminator, mapping network, optimizers)
        - the obtained average results
        - the epoch ID
        - the step/batch ID,
        - the training hyperparameters.
        Returns:
        - the dictionary with the hyperparameters.
        '''
        self.load_config()

        if load_model_from is not None:
            fname = load_model_from
        elif checkpoint_num is not None:
            fname = self.model_name(checkpoint_num)
        else:
            print('Loading model from file failed due to incorrect arguments.')
            return None

        results_loaded = torch.load(fname, weights_only=True)

        if 'version' in results_loaded:
            print(f"Loading model from version {results_loaded['version']}")

        self.step    = results_loaded['step']
        self.epoch   = results_loaded['epoch']
        self.results = results_loaded['results']

        try:
            self.GAN.load_state_dict(results_loaded['GAN'])
        except Exception as e:
            print('Unable to load the save model from file.')
            raise e
        if self.GAN.fp16 and 'amp' in results_loaded:
            amp.load_state_dict(results_loaded['amp'])

        return results_loaded['hyperparameters']

    def save_results_csv(self, version=1):
        '''
        Save training metrics to a CSV file.
        Each metric is a list in the 'results' dictionary.
        '''
        # Make all metrics lists with same length to make it possible to write them
        # to a CSV file. If necessary, append white spaces to the shorter lists.
        aux_g_loss   = self.results.get('g_loss')
        len_g_loss   = len(aux_g_loss)
        max_len      = len_g_loss
        aux_d_loss   = self.results.get('d_loss')
        len_d_loss   = len(aux_d_loss)
        if len_d_loss > max_len:
            max_len  = len_d_loss
        aux_gp       = self.results.get('gp')
        len_gp       = len(aux_gp)
        if len_gp > max_len:
            max_len  = len_gp

        aux_plp      = self.results.get('plp')
        if aux_plp is not None and len(aux_plp) > 0:
            len_plp      = len(aux_plp)
            if len_plp > max_len:
                max_len  = len_plp

        for n in range(len_g_loss, max_len, 1):
            aux_g_loss.append('')
        for n in range(len_d_loss, max_len, 1):
            aux_d_loss.append('')
        for n in range(len_gp, max_len, 1):
            aux_gp.append('')

        if aux_plp is not None and len(aux_plp) > 0:
            for n in range(len_plp, max_len, 1):
                aux_plp.append('')

        if aux_plp is not None and len(aux_plp) > 0:
            columns     = ['g_loss', 'd_loss', 'gp', 'plp']
            results_lst = list(
                zip(
                    aux_g_loss,
                    aux_d_loss,
                    aux_gp,
                    aux_plp,
                )
            )
        else:
            columns     = ['g_loss', 'd_loss', 'gp']
            results_lst = list(
                zip(
                    aux_g_loss,
                    aux_d_loss,
                    aux_gp,
                )
            )

        results_df = pd.DataFrame(
            results_lst,
            columns = columns,
        )
        results_df.head()

        file_name = str(self.results_dir / f'{self.name}_results_{str(version).zfill(2)}.csv')
        f'results/{config["experiment_name"]}_results.csv'
        results_df.to_csv(file_name)

    def print_model_summary(self):
        '''
        Print a summary of the architectures of the style mapping network, the generator
        and the discriminator.
        '''

        if self.GAN is None:
            self.init_GAN()
            is_new_GAN = True
        else:
            is_new_GAN = False

        # Style mapping network summary

        aux_data = torch.randn(self.batch_size, self.latent_dim, device=device)
        sumSM = summary(
            self.GAN.S,
            input_data   = aux_data,
            col_width    = 16,
            col_names    = ["kernel_size", "output_size", "num_params"],
            row_settings = ["var_names"],
        )
        print(sumSM)

        # Generator model summary

        style_aux    = noise_list(self.batch_size, self.GAN.G.num_layers, self.latent_dim, device=self.rank)
        noise_aux    = image_noise(self.batch_size, self.image_size, device=self.rank)
        w_space_aux  = latent_to_w(self.GAN.S, style_aux)
        w_styles_aux = styles_def_to_tensor(w_space_aux)
        w_shape      = [w_styles_aux.shape[0], w_styles_aux.shape[1], w_styles_aux.shape[2]]
        # w_styles_aux shape: [32, 6, 512]
        # noise_aux shape:    [32, 6, 512]

        sumG = summary(
            self.GAN.G,
            input_data   = [w_styles_aux, noise_aux],
            col_width    = 16,
            col_names    = ["kernel_size", "output_size", "num_params"],
            row_settings = ["var_names"],
        )
        print(sumG)
        del w_styles_aux
        del w_space_aux
        del noise_aux
        del style_aux

        # Discriminator model summary

        aux_data = torch.randn(
            (
            self.batch_size,
            self.channels,
            self.image_size,
            self.image_size
            )).cuda(self.rank)

        sumD = summary(
            self.GAN.D,
            input_data   = aux_data,
            col_width    = 16,
            col_names    = ["kernel_size", "output_size", "num_params"],
            row_settings = ["var_names"],
        )
        print(sumD)
        del aux_data

        if is_new_GAN == True:
            del self.GAN
            self.GAN = None

    def save_model_onnx(self):
        '''
        Export the StyleGAN2 model to ONNX format.
        '''
        file_save_onnx_S = str(self.models_dir / f'{self.name}_map_network_architecture.onnx')
        file_save_onnx_G = str(self.models_dir / f'{self.name}_generator_architecture.onnx')
        file_save_onnx_D = str(self.models_dir / f'{self.name}_discriminator_architecture.onnx')

        if self.GAN is None:
            print("Creating a StyleGAN2 model in save_model_onnx function ...")
            self.init_GAN()
            self.GAN.cuda(self.rank)
            is_new_GAN = True
        else:
            is_new_GAN = False

        self.GAN.eval()

        # Export the style mapping network ..........................................

        aux_data_S     = torch.randn(self.batch_size, self.latent_dim, device=device)
        input_names_S  = [ "z" ]
        output_names_S = [ "w" ]
        torch.onnx.export(
            self.GAN.S,
            aux_data_S,
            file_save_onnx_S,
            verbose       = False,
            input_names   = input_names_S,
            output_names  = output_names_S,
            export_params = True,
        )

        # Export the discriminator ..................................................

        aux_data_D = torch.randn(
            self.batch_size,
            self.channels,
            self.image_size,
            self.image_size,
            device=device
            )
        input_names_D  = [ "image" ]
        output_names_D = [ "real_fake" ]
        torch.onnx.export(
            self.GAN.D,
            aux_data_D,
            file_save_onnx_D,
            verbose       = False,
            input_names   = input_names_D,
            output_names  = output_names_D,
            export_params = True,
        )

        # Export the generator .....................................................

        style_aux      = noise_list(self.batch_size, self.GAN.G.num_layers, self.latent_dim, device=self.rank)
        noise_aux      = image_noise(self.batch_size, self.image_size, device=self.rank)
        w_space_aux    = latent_to_w(self.GAN.S, style_aux)
        w_styles_aux   = styles_def_to_tensor(w_space_aux)
        aux_data_G     = (w_styles_aux, noise_aux)
        input_names_G  = [ "w", "noise" ]
        output_names_G = [ "generated_image" ]
        torch.onnx.dynamo_export(
            self.GAN.G,
            *aux_data_G,
        ).save(file_save_onnx_G)

        if is_new_GAN == True:
            print("Deleting the created StyleGAN2 model in save_model_onnx function ...")
            del self.GAN
            self.GAN = None

# Training loop functions

In [ ]:
def train_stylegan2(
        rank,
        world_size,
        config,
        start_epoch=0,
    ):
    '''
    Function used to train the StyleGAN2 model from the beginning 
    or to continue training of a saved model.
    '''
    is_main = (rank == 0)
    is_ddp  = (world_size > 1)

    # If training distributedly in multiple GPUs
    if is_ddp:
        set_seed(config['seed'])
        os.environ['MASTER_ADDR'] = 'localhost'
        os.environ['MASTER_PORT'] = '12355'
        dist.init_process_group('nccl', rank=rank, world_size=world_size)
        print(f"{rank + 1}/{world_size} process initialized.")

    # Update configuration dictionary with 'is_ddp', 'rank', 'world_size'
    config.update({'is_ddp': is_ddp})
    config.update({'rank': rank})
    config.update({'world_size': world_size})

    # Create a training session .....................................................

    train_session = Trainer(start_epoch, **config)

    # Load a previously saved model from file .......................................
    if config['load_trained_model']:
        print('Loading a saved model from file ...')
        _ = train_session.load(load_model_from=config['load_model_from'])
        print(f'Training restarts in epoch {train_session.epoch+1} step {train_session.step+1}')
    else:
        print('Initializing the training session ...')

        if config['delete_folders_if_exist'] == True:
            # CAUTION: THIS ACTION REMOVES ALL PREVIOUSLY PRODUCED FILES
            #          FOR AN EXPERIMENT WITH THE SAME NAME
            train_session.clear()

    # Setup Dataset and DataLoader for training .....................................

    train_session.setup_dataset()

    # Verify if Dataset and DataLoader work correctly ...............................

    train_session.check_dataloader()

    # Setup dataset dependent hyperparameters .......................................

    dataset_len                    = len(train_session.loader)
    config['train_steps']          = int((config['epochs'] * dataset_len) / config['gradient_accumulate_every'])
    config['save_every_steps']     = int((config['checkp_interval'] * dataset_len) / config['gradient_accumulate_every'])
    train_session.save_every_steps = config['save_every_steps']

    print(f'Dataset length: {dataset_len} batches of {config["batch_size"]} images each')

    # Print training configuration .................................................

    print('Configuration parameters:')
    for key, value in config.items():
        print(f'\t{key}: {value}')

    # Print models' architectures ..................................................

    train_session.print_model_summary()

    # Export the model in ONNX format ..............................................
    #
    # train_session.save_model_onnx()

    # Iterate over all the training steps ..........................................

    for epoch in range(train_session.epoch, config['epochs']):

        ts  = time.time()

        loop = tqdm(train_session.loader, leave=True)

        for batch_idx, real in enumerate(loop):

            retry_call(train_session.train_step, tries=3, exceptions=NanException)
            if is_main and (train_session.step+1) % train_session.log_interval == 0:
                train_session.print_metrics(loop)

            train_session.step += 1

        te        = time.time()
        texec_sec = te - ts
        texec_str = time_format(texec_sec)
        print(f'Epoch {epoch+1} training time: {texec_str}')
        train_session.results['epoch_training_time'].append(texec_sec)

        train_session.epoch = epoch + 1

    # Save the final checkpoint ..................................................
    train_session.save(train_session.checkpoint_num, config)

    # Save the obtained results to a CSV file ....................................
    train_session.save_results_csv(1)

    if is_ddp:
        dist.destroy_process_group()


In [ ]:
def train_stylegan2_from_folder(
    config,
    ):
    '''
    Top level function used to train the StyleGAN2 model, to generate images with 
    a trained model, or to generate a sequence of images from latents that are obtained 
    with the interpolation between two starting latents.
    '''
    # Generate grid(s) of images with a trained model .....................................
    if config['generate']:

        # Update configuration dictionary with 'is_ddp', 'rank', 'world_size'
        config.update({'is_ddp': False})
        config.update({'rank': 0})
        config.update({'world_size': 1})
        epoch = 0

        train_session = Trainer(epoch, **config)
        _             = train_session.load(load_model_from=config['load_model_from'])
        samples_name  = timestamped_filename()
        for num in tqdm(range(config['num_grids_generate'])):
            train_session.evaluate(
                train_session.epoch,
                train_session.step,
                postfix = f'-{samples_name}-{num}',
            )
        print(f'Sample images generated at {train_session.results_dir}//{samples_name}')
        return

    # Generate an interpolation videos between two latents ................................
    if config['generate_interpolation']:

        # Update configuration dictionary with 'is_ddp', 'rank', 'world_size'
        config.update({'is_ddp': False})
        config.update({'rank': 0})
        config.update({'world_size': 1})
        epoch = 0

        train_session = Trainer(epoch, **config)
        _             = train_session.load(load_model_from=config['load_model_from'])
        samples_name  = timestamped_filename()
        train_session.generate_interpolation(
            samples_name,
            config['grid_size'],
            num_steps   = config['interpolation_steps'],
            save_frames = config['save_frames'],
        )
        print(f'Interpolation generated at {results_dir}/{samples_name}')
        return

    # Train the StyleGAN2 model ..........................................................
    world_size = torch.cuda.device_count()
    print(f'GPU count: {world_size}')

    # Training in a single GPU ................................
    if world_size == 1 or not config['multi_gpus']:
        train_stylegan2(
            0,      # rank
            1,      # world size
            config,
        )
        return

    # Distributed training in multiple GPUs ..................
    mp.spawn(
        train_stylegan2,
        args = (
            world_size,
            config
        ),
        nprocs = world_size,
        join   = True,
    )

## Train the StyleGAN2 model

In [ ]:
train_stylegan2_from_folder(config)

In [ ]:
# Mark the Weights & Bias run as finished
wandb.finish()

## Conclusion

In this notebook, we make a clean, simple, and readable implementation from scratch for a huge project which is StyleGAN2 using PyTorch. we try to replicate the original paper as closely as possible.